# Módulo 4: Bases de datos - Ejercicio de evaluación

## 1. Creamos el comando para crear la base de datos y sus tablas en el archivo SQL:

In [9]:
import mysql.connector as con
import random
from datetime import datetime, timedelta

# Configuración de la conexión a MySQL
# He definido una función para establecer y retornar la conexión con la base de datos MySQL.
def get_connection():
    return con.connect(
        host="localhost",
        port="3306",
        user="root",
        password="admin",
        database="supermercado"
    )
    
# He creado una función para verificar si la base de datos 'supermercado' existe y, en caso contrario, crearla.
def create_database():
    connection = con.connect(
        host="localhost",
        port="3306",
        user="root",
        password="admin"
    )
    cursor = connection.cursor()
    cursor.execute("SHOW DATABASES;")  # Obtengo las bases de datos existentes
    databases = [db[0] for db in cursor.fetchall()]  # Genero una lista de bases de datos
    if "supermercado" not in databases:  # Si la base de datos no existe, la creo
        cursor.execute("CREATE DATABASE supermercado DEFAULT CHARACTER SET utf8;")
        print("Base de datos creada exitosamente.")
    else:
        print("La base de datos ya existe.")
    cursor.close()
    connection.close()

# He creado una función que define y crea las tablas necesarias en la base de datos 'supermercado'.
def create_tables():
    connection = get_connection()
    cursor = connection.cursor()
    
    # Defino los esquemas de cada tabla con las relaciones necesarias.
    tables = {
        "tiendas": """
            CREATE TABLE IF NOT EXISTS tiendas (
                id_tienda INT AUTO_INCREMENT PRIMARY KEY,
                nombre_tienda VARCHAR(100) NOT NULL,
                direccion VARCHAR(255) NOT NULL,
                ciudad VARCHAR(100) NOT NULL
            );
        """,
        "empleados": """
            CREATE TABLE IF NOT EXISTS empleados (
                id_empleado INT AUTO_INCREMENT PRIMARY KEY,
                nombre_empleado VARCHAR(100) NOT NULL,
                puesto ENUM('Cajero', 'Gerente', 'Reponedor', 'Vendedor') NOT NULL,
                id_tienda INT,
                FOREIGN KEY (id_tienda) REFERENCES tiendas(id_tienda) ON DELETE SET NULL
            );
        """,
        "categorias": """
            CREATE TABLE IF NOT EXISTS categorias (
                id_categoria INT AUTO_INCREMENT PRIMARY KEY,
                nombre_categoria VARCHAR(100) NOT NULL
            );
        """,
        "productos": """
            CREATE TABLE IF NOT EXISTS productos (
                id_producto INT AUTO_INCREMENT PRIMARY KEY,
                nombre_producto VARCHAR(100) NOT NULL,
                precio DECIMAL(10,2) NOT NULL,
                stock INT NOT NULL,
                id_categoria INT,
                FOREIGN KEY (id_categoria) REFERENCES categorias(id_categoria) ON DELETE SET NULL
            );
        """,
        "clientes": """
            CREATE TABLE IF NOT EXISTS clientes (
                id_cliente INT AUTO_INCREMENT PRIMARY KEY,
                first_name VARCHAR(100) NOT NULL,
                last_name VARCHAR(100) NOT NULL,
                email VARCHAR(150) UNIQUE NOT NULL,
                telefono VARCHAR(15) NOT NULL,
                codigo_postal INT,
                direccion VARCHAR(255) NOT NULL
            );
        """,
        "ordenes": """
            CREATE TABLE IF NOT EXISTS ordenes (
                id_orden INT AUTO_INCREMENT PRIMARY KEY,
                id_cliente INT NOT NULL,
                id_empleado INT NOT NULL,
                fecha_orden DATETIME DEFAULT CURRENT_TIMESTAMP,
                metodo_pago ENUM('Tarjeta', 'Efectivo') NOT NULL,
                FOREIGN KEY (id_cliente) REFERENCES clientes(id_cliente) ON DELETE CASCADE,
                FOREIGN KEY (id_empleado) REFERENCES empleados(id_empleado) ON DELETE CASCADE
            );
        """,
        "detalle_orden": """
            CREATE TABLE IF NOT EXISTS detalle_orden (
                id_detalle INT AUTO_INCREMENT PRIMARY KEY,
                id_orden INT NOT NULL,
                id_producto INT NOT NULL,
                cantidad INT NOT NULL,
                precio_unitario DECIMAL(10,2) NOT NULL,
                descuento DECIMAL(5,2) DEFAULT 0.00,
                FOREIGN KEY (id_orden) REFERENCES ordenes(id_orden) ON DELETE CASCADE,
                FOREIGN KEY (id_producto) REFERENCES productos(id_producto) ON DELETE CASCADE
            );
        """
    }

    # Itero sobre el diccionario de tablas y ejecuto las consultas para crearlas.
    for table, query in tables.items():
        cursor.execute(query)
    connection.commit()
    cursor.close()
    connection.close()
    print("Tablas creadas correctamente.")

 ## 2: Generar datos demo desde Python

In [10]:
# He definido una función para insertar datos en cualquier tabla en lotes para mejorar la eficiencia.
def insert_data(table, columns, values, batch_size=1000):
    connection = get_connection()
    cursor = connection.cursor()
    try:
        placeholders = ", ".join(["%s"] * len(columns))
        column_names = ", ".join(columns)
        query = f"REPLACE INTO {table} ({column_names}) VALUES ({placeholders});"
        
        for i in range(0, len(values), batch_size):
            batch = values[i:i+batch_size]
            cursor.executemany(query, batch)
            connection.commit()
        print(f"Datos insertados en {table} correctamente.")
    except Exception as e:
        connection.rollback()
        print(f"Error en {table}: {e}")
    finally:
        cursor.close()
        connection.close()

def generate_tiendas():
    # Genero datos aleatorios para la tabla 'tiendas'.
    tiendas = [(f"Super {nombre}", f"Calle {i} Centro", random.choice(["Madrid", "Barcelona", "México DF", "Tijuana", "Tokio", "Osaka", "Casablanca"])) 
               for i, nombre in enumerate(["Norte", "Sur", "Este", "Oeste", "Centro"], 1)]
    insert_data("tiendas", ["nombre_tienda", "direccion", "ciudad"], tiendas)

def generate_empleados():
    # Genero datos aleatorios para la tabla 'empleados'.
    empleados = []
    for i in range(1, 21):
        for tienda_id in range(1, 6):  # 5 tiendas
            nombre_empleado = f"Empleado {i}"
            puesto = random.choice(['Cajero', 'Gerente', 'Reponedor', 'Vendedor'])
            empleados.append((nombre_empleado, puesto, tienda_id))
    insert_data("empleados", ["nombre_empleado", "puesto", "id_tienda"], empleados)

def generate_categorias():
    # Genero datos aleatorios para la tabla 'categorias'.
    categorias = ["Lácteos", "Carnes", "Frutas", "Verduras", "Bebidas", "Snacks", "Limpieza", "Medicina", "Panadería", "Tecnología"]
    categorias_data = [(categoria,) for categoria in categorias]
    insert_data("categorias", ["nombre_categoria"], categorias_data)

def generate_productos():
    # Genero datos aleatorios para la tabla 'productos'.
    productos = []
    for categoria_id in range(1, 11):  # 10 categorías
        for i in range(1, 5):  # 4 productos por categoría
            nombre_producto = f"Producto {i} de Categoria {categoria_id}"
            precio = round(random.uniform(0.5, 50.0), 2)
            stock = random.randint(0, 500)
            productos.append((nombre_producto, precio, stock, categoria_id))
    insert_data("productos", ["nombre_producto", "precio", "stock", "id_categoria"], productos)

def generate_clientes():
    # Genero datos aleatorios para la tabla 'clientes'.
    clientes = []
    calles = ["Calle Falsa", "Avenida Siempre Viva", "Calle Mayor", "Paseo del Prado", "Gran Vía"]
    
    for i in range(1, 2001):  # 2000 clientes
        first_name = f"Cliente {i} First"
        last_name = f"Cliente {i} Last"
        email = f"cliente{i}@test.com"
        telefono = f"{random.randint(600000000, 699999999)}"  # Números de teléfono ficticios
        codigo_postal = f"{random.randint(10000, 99999)}"
        direccion = f"{random.choice(calles)} {random.randint(1, 300)}"  # Dirección aleatoria 
        clientes.append((first_name, last_name, email, telefono, direccion, codigo_postal))
    insert_data("clientes", ["first_name", "last_name", "email", "telefono", "direccion", "codigo_postal"], clientes)

def generate_ordenes():
    # Genera datos aleatorios para la tabla 'ordenes'.
    ordenes = []
    clientes_ids = [i for i in range(1, 2001)]  # 2000 clientes insertados previamente
    empleados_ids = [i for i in range(1, 101)]  # 100 empleados insertados previamente

    for i in range(1, 10001):  # 10000 ordenes
        id_cliente = random.choice(clientes_ids)  # Cliente ya insertado
        id_empleado = random.choice(empleados_ids)  # Empleado ya insertado
        fecha_orden = generate_random_dates("2024-01-01", "2025-01-01", 1)[0]
        metodo_pago = random.choice(['Tarjeta', 'Efectivo'])
        ordenes.append((id_cliente, id_empleado, fecha_orden, metodo_pago))

    insert_data("ordenes", ["id_cliente", "id_empleado", "fecha_orden", "metodo_pago"], ordenes)

def generate_detalle_orden():
    # Genera datos aleatorios para la tabla 'detalle_orden'.
    detalle_orden = []
    productos_ids = [i for i in range(1, 41)]
    ordenes_ids = [i for i in range(1, 10001)]

    for i in range(1, 30001):  # 30000 detalles de orden
        id_orden = random.choice(ordenes_ids)  # Orden ya insertada
        id_producto = random.choice(productos_ids)  # Producto ya insertado
        cantidad = random.randint(1, 20)
        precio_unitario = round(random.uniform(0.5, 50.0), 2)
        descuento = random.choice([0.00, round(random.uniform(0.5, 5.0), 2)])
        detalle_orden.append((id_orden, id_producto, cantidad, precio_unitario, descuento))

    insert_data("detalle_orden", ["id_orden", "id_producto", "cantidad", "precio_unitario", "descuento"], detalle_orden)

def generate_random_dates(start_date, end_date, num_dates):
    # Genera fechas aleatorias entre un rango de fechas.
    start_timestamp = datetime.strptime(start_date, "%Y-%m-%d")
    end_timestamp = datetime.strptime(end_date, "%Y-%m-%d")
    dates = []
    for _ in range(num_dates):
        random_date = start_timestamp + (end_timestamp - start_timestamp) * random.random()
        dates.append(random_date)
    return dates

# Defino una función para eliminar todos los registros de las tablas antes de insertar nuevos datos.
def clear_tables():
    tables = ["tiendas", "empleados", "categorias", "productos", "clientes", "ordenes", "detalle_orden"]
    connection = get_connection()
    cursor = connection.cursor()
    try:
        cursor.execute("SET FOREIGN_KEY_CHECKS = 0;")
        for table in tables:
            query = f"TRUNCATE TABLE {table};"
            cursor.execute(query)
        connection.commit()
        cursor.execute("SET FOREIGN_KEY_CHECKS = 1;")
    except Exception as e:
        connection.rollback()
        print(f"Error al limpiar las tablas: {e}")
    finally:
        cursor.close()
        connection.close()

# Resultado:

In [12]:
# Llamada para crear la base de datos y las tablas
create_database()
create_tables()
# Limpio las tablas para que se inserten bien los datos en caso de que haya datos existentes
clear_tables()
# Ahora comprobaré si se generan bien los datos
generate_tiendas()
generate_empleados()
generate_categorias()
generate_productos()
generate_clientes()
generate_ordenes()
generate_detalle_orden()

Base de datos creada exitosamente.
Tablas creadas correctamente.
Datos insertados en tiendas correctamente.
Datos insertados en empleados correctamente.
Datos insertados en categorias correctamente.
Datos insertados en productos correctamente.
Datos insertados en clientes correctamente.
Datos insertados en ordenes correctamente.
Datos insertados en detalle_orden correctamente.


## 3: Consultas SQL

### 1 - Listado de órdenes con detalles de cliente y empleado:

```sql
SELECT 
    o.id_orden, 
    o.fecha_orden, 
    c.first_name AS cliente_nombre, 
    c.last_name AS cliente_apellido, 
    e.nombre_empleado, 
    o.metodo_pago
FROM 
    ordenes o
JOIN 
    clientes c ON o.id_cliente = c.id_cliente
JOIN 
    empleados e ON o.id_empleado = e.id_empleado;

```
### 2 - Productos con stock bajo (menor a 10):

```sql
SELECT 
    p.nombre_producto, 
    ca.nombre_categoria, 
    p.stock
FROM 
    productos p
JOIN 
    categorias ca ON p.id_categoria = ca.id_categoria
WHERE 
    p.stock < 10;
```

### 3 - Ventas totales por categoría:

```sql
SELECT 
    ca.nombre_categoria, 
    ROUND(SUM(do.cantidad * do.precio_unitario)) AS ventas_totales
FROM 
    detalle_orden do
JOIN 
    productos p ON do.id_producto = p.id_producto
JOIN 
    categorias ca ON p.id_categoria = ca.id_categoria
GROUP BY 
    ca.id_categoria;
```

### 4 - Clientes con mayores gastos acumulados:

```sql
SELECT 
    c.first_name AS cliente_nombre, 
    c.last_name AS cliente_apellido, 
    SUM(do.cantidad * do.precio_unitario - IFNULL(do.descuento, 0)) AS gasto_total
FROM 
    detalle_orden do
JOIN 
    ordenes o ON do.id_orden = o.id_orden
JOIN 
    clientes c ON o.id_cliente = c.id_cliente
GROUP BY 
    c.id_cliente
ORDER BY 
    gasto_total DESC;

```

### 5 - Empleados y número de órdenes gestionadas:

```sql
SELECT 
    e.nombre_empleado, 
    e.puesto, 
    COUNT(o.id_orden) AS num_ordenes
FROM 
    empleados e
LEFT JOIN 
    ordenes o ON e.id_empleado = o.id_empleado
GROUP BY 
    e.id_empleado;
```

### 6 - Órdenes filtradas por fecha y tienda:

```sql
SELECT 
    o.id_orden, 
    o.fecha_orden, 
    t.nombre_tienda, 
    c.first_name AS cliente_nombre, 
    c.last_name AS cliente_apellido
FROM 
    ordenes o
JOIN 
    tiendas t ON o.id_empleado = t.id_tienda
JOIN 
    clientes c ON o.id_cliente = c.id_cliente
WHERE 
    o.fecha_orden BETWEEN '2025-01-01' AND '2025-01-31'
    AND t.id_tienda = 1;
```

### 7 - Ranking de productos más vendidos en cada tienda:

```sql
SELECT 
    t.nombre_tienda, 
    p.nombre_producto, 
    SUM(do.cantidad) AS total_vendido
FROM 
    detalle_orden do
JOIN 
    ordenes o ON do.id_orden = o.id_orden
JOIN 
    empleados e ON o.id_empleado = e.id_empleado
JOIN 
    tiendas t ON e.id_tienda = t.id_tienda
JOIN 
    productos p ON do.id_producto = p.id_producto
GROUP BY 
    t.id_tienda, p.id_producto
ORDER BY 
    t.id_tienda, total_vendido DESC
LIMIT 3;
```

### 8 - Total de ventas por cliente considerando los productos vendidos:

```sql
SELECT 
    c.first_name AS cliente_nombre, 
    c.last_name AS cliente_apellido, 
    FLOOR(
        (SELECT SUM(do.cantidad * do.precio_unitario - IFNULL(do.descuento, 0))
         FROM detalle_orden do
         JOIN ordenes o ON do.id_orden = o.id_orden
         WHERE o.id_cliente = c.id_cliente)
    ) AS total_ventas
FROM 
    clientes c
ORDER BY 
    total_ventas DESC;
```

cursor.execute('''
select * from customers;
''')